# 2층 신경망 역전파 유도

이 노트북은 입력 2개 → 은닉 2개 → 출력 1개인 2층 신경망에서

1. 순전파 수식을 쓰고 (shape 표기 포함),
2. 출력층에서 입력층 방향으로 **연쇄 법칙**을 적용해 역전파 수식을 단계별로 유도하고,
3. 고정된 숫자 예제로 **손계산**한 뒤,
4. 같은 계산을 **NumPy**로 수행해 손계산과 소수점 4자리까지 일치하는지 확인하고,
5. `src/calculus.py`의 **중심차분**으로 기울기를 한 번 더 검증(gradient check)한다.

| 기호 | 뜻 | shape |
|---|---|---|
| $x$ | 입력 | (2,) |
| $W_1$, $b_1$ | 은닉층 가중치, 편향 | (2,2), (2,) |
| $z_1$, $a_1$ | 은닉층 선형 결합, 활성화 출력 | (2,), (2,) |
| $W_2$, $b_2$ | 출력층 가중치, 편향 | (2,), scalar |
| $z_2$, $\hat{y}$ | 출력층 선형 결합, 예측값 | scalar, scalar |
| $y$ | 정답 | scalar |
| $L$ | 손실 (BCE) | scalar |

## 1. 순전파 수식

$$
\begin{aligned}
z_1 &= W_1 x + b_1 &\quad& (2,2) @ (2,) + (2,) \to (2,) \\
a_1 &= \sigma(z_1) && (2,) \text{ (원소별 적용)} \\
z_2 &= W_2 \cdot a_1 + b_2 && (2,) \cdot (2,) + \text{scalar} \to \text{scalar} \\
\hat{y} &= \sigma(z_2) && \text{scalar} \\
L &= -\left[\, y \log \hat{y} + (1-y) \log (1-\hat{y}) \,\right] && \text{scalar}
\end{aligned}
$$

활성화 함수 sigmoid와 그 미분은 다음과 같다.

$$
\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \sigma'(z) = \sigma(z)\,\bigl(1 - \sigma(z)\bigr)
$$

**sigmoid 미분 유도**: $\sigma(z) = (1 + e^{-z})^{-1}$ 을 미분하면

$$
\sigma'(z) = \frac{e^{-z}}{(1 + e^{-z})^2}
= \underbrace{\frac{1}{1 + e^{-z}}}_{\sigma(z)} \cdot \underbrace{\frac{e^{-z}}{1 + e^{-z}}}_{1 - \sigma(z)}
= \sigma(z)\bigl(1 - \sigma(z)\bigr)
$$

## 2. 역전파 수식 유도 (출력층 → 입력층)

**연쇄 법칙**: 여러 단계를 거친 영향은 각 단계의 영향(미분)을 곱한 것이다. 출력 쪽에서부터 거꾸로 계산하면 앞에서 구한 값을 다음 단계에서 그대로 재사용할 수 있다.

계산 그래프: $x \xrightarrow{W_1, b_1} z_1 \xrightarrow{\sigma} a_1 \xrightarrow{W_2, b_2} z_2 \xrightarrow{\sigma} \hat{y} \xrightarrow{\text{BCE}} L$

### ① $\partial L / \partial \hat{y}$ — shape: scalar

$$
\frac{\partial L}{\partial \hat{y}} = -\left( \frac{y}{\hat{y}} - \frac{1-y}{1-\hat{y}} \right) = -\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}}
$$

($\log$ 의 미분은 $\frac{d}{du}\log u = \frac{1}{u}$, $\frac{d}{d\hat{y}}\log(1-\hat{y}) = -\frac{1}{1-\hat{y}}$)

### ② $\partial L / \partial z_2$ — shape: scalar  ← 체크포인트

$$
\frac{\partial L}{\partial z_2} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_2}
= \left( -\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}} \right) \cdot \hat{y}(1-\hat{y})
= -y(1-\hat{y}) + (1-y)\hat{y}
= \hat{y} - y
$$

sigmoid와 BCE를 함께 쓰면 복잡한 분수가 모두 지워지고 **"예측 − 정답"** 이라는 아주 단순한 결과가 된다.

### ③ $\partial L / \partial W_2$ — shape: (2,),  $\partial L / \partial b_2$ — shape: scalar

$z_2 = W_2 \cdot a_1 + b_2 = W_{2,1} a_{1,1} + W_{2,2} a_{1,2} + b_2$ 이므로 $\frac{\partial z_2}{\partial W_{2,i}} = a_{1,i}$, $\frac{\partial z_2}{\partial b_2} = 1$.

$$
\frac{\partial L}{\partial W_2} = \frac{\partial L}{\partial z_2} \cdot a_1, \qquad
\frac{\partial L}{\partial b_2} = \frac{\partial L}{\partial z_2}
$$

### ④ $\partial L / \partial a_1$ — shape: (2,)

같은 식에서 $\frac{\partial z_2}{\partial a_{1,i}} = W_{2,i}$ 이므로

$$
\frac{\partial L}{\partial a_1} = \frac{\partial L}{\partial z_2} \cdot W_2
$$

### ⑤ $\partial L / \partial z_1$ — shape: (2,)  ← 체크포인트

$a_1 = \sigma(z_1)$ 은 원소별로 적용되므로, 원소별 곱 $\odot$ 을 쓴다.

$$
\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \odot \sigma'(z_1) = \frac{\partial L}{\partial a_1} \odot a_1 \odot (1 - a_1)
$$

### ⑥ $\partial L / \partial W_1$ — shape: (2,2),  $\partial L / \partial b_1$ — shape: (2,)

$z_{1,i} = \sum_j W_{1,ij}\, x_j + b_{1,i}$ 이므로 $\frac{\partial z_{1,i}}{\partial W_{1,ij}} = x_j$. 즉 $(i, j)$ 칸의 기울기는 $\frac{\partial L}{\partial z_{1,i}} \cdot x_j$ 이고, 이것을 표로 모으면 **외적(outer product)** 이다.

$$
\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial z_1} \otimes x = \frac{\partial L}{\partial z_1}\, x^{\top}
\quad (2,) \otimes (2,) \to (2,2), \qquad
\frac{\partial L}{\partial b_1} = \frac{\partial L}{\partial z_1}
$$

**shape 확인**: 각 기울기의 shape은 대응하는 변수의 shape과 같아야 한다. ($\partial L/\partial W_1$: (2,2) = $W_1$: (2,2))

## 3. 손계산 (고정 예제)

| 변수 | 값 | shape |
|---|---|---|
| $x$ | [1, 0] | (2,) |
| $W_1$ | [[0.1, 0.2], [0.3, 0.4]] | (2,2) |
| $b_1$ | [0, 0] | (2,) |
| $W_2$ | [0.5, 0.6] | (2,) |
| $b_2$ | 0 | scalar |
| $y$ (정답) | 1 | scalar |

중간값은 소수점 6자리로 계산하고, 결과는 4자리로 반올림해 기록한다. (중간값을 4자리로 먼저 반올림하면 $z_2$ 가 0.6071로 나와 최종값과 0.0001 차이가 생길 수 있다.)

### 순전파

| 단계 | 계산 | 결과 | shape |
|---|---|---|---|
| $z_1 = W_1 x + b_1$ | [0.1×1 + 0.2×0, 0.3×1 + 0.4×0] | **[0.1000, 0.3000]** | (2,) |
| $a_1 = \sigma(z_1)$ | [1/(1+e^{−0.1}), 1/(1+e^{−0.3})] = [1/1.904837, 1/1.740818] = [0.524979, 0.574443] | **[0.5250, 0.5744]** | (2,) |
| $z_2 = W_2 \cdot a_1 + b_2$ | 0.5×0.524979 + 0.6×0.574443 = 0.262490 + 0.344666 | **0.6072** (0.607155) | scalar |
| $\hat{y} = \sigma(z_2)$ | 1/(1+e^{−0.607155}) = 1/1.544918 | **0.6473** (0.647292) | scalar |
| $L$ | −log(0.647292) | **0.4350** (0.434958) | scalar |

### 역전파

| 단계 | 계산 | 결과 | shape |
|---|---|---|---|
| ① $\partial L/\partial \hat{y}$ | −1/0.647292 (y = 1이라 두 번째 항은 0) | **−1.5449** | scalar |
| ② $\partial L/\partial z_2$ | $\hat{y} - y$ = 0.647292 − 1 (확인: −1.544899 × 0.647292 × 0.352708 = −0.352708) | **−0.3527** | scalar |
| ③ $\partial L/\partial W_2$ | −0.352708 × [0.524979, 0.574443] | **[−0.1852, −0.2026]** | (2,) |
| ③ $\partial L/\partial b_2$ | = $\partial L/\partial z_2$ | **−0.3527** | scalar |
| ④ $\partial L/\partial a_1$ | −0.352708 × [0.5, 0.6] | **[−0.1764, −0.2116]** | (2,) |
| ⑤ $\sigma'(z_1)$ | [0.524979 × 0.475021, 0.574443 × 0.425557] = [0.249376, 0.244458] | [0.2494, 0.2445] | (2,) |
| ⑤ $\partial L/\partial z_1$ | [−0.176354 × 0.249376, −0.211625 × 0.244458] | **[−0.0440, −0.0517]** | (2,) |
| ⑥ $\partial L/\partial W_1$ | outer([−0.043979, −0.051733], [1, 0]) | **[[−0.0440, 0], [−0.0517, 0]]** | (2,2) |
| ⑥ $\partial L/\partial b_1$ | = $\partial L/\partial z_1$ | **[−0.0440, −0.0517]** | (2,) |

> **과제 예시와 다른 점**: 과제 PDF의 예시는 $z_2$ = 0.6071, $\partial L/\partial z_1$ = [−0.0440, **−0.0513**] 이다.
> - $z_2$: 정확한 값은 0.607155라서 4자리 반올림하면 0.6072다. 0.6071은 $a_1$ 을 먼저 4자리로 반올림(0.5×0.5250 + 0.6×0.5744 = 0.60714)했을 때 나온다.
> - $\partial L/\partial z_1$ 의 두 번째 값: −0.211625 × 0.244458 = **−0.0517** 이다. 아래 NumPy 계산과 중심차분 검증 모두 −0.0517을 가리킨다.
>
> $x$ 의 두 번째 성분이 0이라 $\partial L/\partial W_1$ 의 두 번째 열은 모두 0이다. 입력이 0이면 그 가중치는 손실에 영향을 주지 않기 때문이다.

## 4. NumPy로 같은 계산하기

계산과 검증 코드는 [`src/backprop.py`](../src/backprop.py)에 있고, 이 노트북은 그 모듈을 불러 쓴다. `uv run python src/backprop.py`로 노트북 없이 같은 검증을 실행할 수도 있다. 아래에 `forward`, `backward` 함수의 코드를 그대로 출력해 두었다.

In [1]:
import inspect
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path.cwd().parent / "src"))  # src/ 모듈을 불러오기 위해
from backprop import (B1, B2, HAND_RESULTS, W1, W2, X, Y_TRUE, backward, compare_with_hand,
                      forward, gradient_check, run_fixed_example)

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

# 고정 예제 (src/backprop.py에 정의)
print("x  =", X, "  shape", X.shape)
print("W1 =", W1.tolist(), "  shape", W1.shape)
print("b1 =", B1, "  shape", B1.shape)
print("W2 =", W2, "  shape", W2.shape)
print("b2 =", B2, "  scalar")
print("y  =", Y_TRUE, "  scalar")

x  = [1. 0.]   shape (2,)
W1 = [[0.1, 0.2], [0.3, 0.4]]   shape (2, 2)
b1 = [0. 0.]   shape (2,)
W2 = [0.5 0.6]   shape (2,)
b2 = 0.0   scalar
y  = 1.0   scalar


In [2]:
print(inspect.getsource(forward))
print(inspect.getsource(backward))

cache, grads = run_fixed_example()

def forward(x, W1, b1, W2, b2, y):
    """2층 신경망 순전파.

    Args:
        x: 입력 (2,). W1: (2, 2). b1: (2,). W2: (2,). b2: scalar. y: 정답 scalar.

    Returns:
        중간값 dict: z1 (2,), a1 (2,), z2 (scalar), y_pred (scalar), loss (scalar).
    """
    z1 = W1 @ x + b1                 # (2,2) @ (2,) + (2,) -> (2,)
    a1 = sigmoid(z1)                 # (2,)
    z2 = W2 @ a1 + b2                # (2,) · (2,) + scalar -> scalar
    y_pred = sigmoid(z2)             # scalar
    loss = -(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))  # BCE, scalar
    return {"z1": z1, "a1": a1, "z2": z2, "y_pred": y_pred, "loss": loss}

def backward(x, W2, y, cache):
    """2층 신경망 역전파. 손계산 기록의 ①~⑥을 그대로 옮긴다.

    Args:
        x: 입력 (2,). W2: (2,). y: 정답 scalar. cache: forward()의 반환값.

    Returns:
        기울기 dict. 각 기울기의 shape은 대응하는 변수의 shape과 같다.
    """
    a1, y_pred = cache["a1"], cache["y_pred"]
    dL_dy_pred = -y / y_pred + (1 - y) / (1 - y_pred)   # ① scalar
    dL_dz2 = y_pred - y             

### 손계산과 비교 (소수점 4자리)

In [3]:
# 위 3절 손계산 결과 (4자리 반올림). src/backprop.py의 HAND_RESULTS에 같은 값이 들어 있다.
for k, v in HAND_RESULTS.items():
    print(f"  {k:<11} = {v}")
print()

all_ok = compare_with_hand({**cache, **grads}, decimals=4)
assert all_ok

  z1          = [0.1, 0.3]
  a1          = [0.525, 0.5744]
  z2          = 0.6072
  y_pred      = 0.6473
  loss        = 0.435
  dL/dy_pred  = -1.5449
  dL/dz2      = -0.3527
  dL/dW2      = [-0.1852, -0.2026]
  dL/db2      = -0.3527
  dL/da1      = [-0.1764, -0.2116]
  dL/dz1      = [-0.044, -0.0517]
  dL/dW1      = [[-0.044, 0.0], [-0.0517, 0.0]]
  dL/db1      = [-0.044, -0.0517]

z1          손계산 [0.1, 0.3]                     NumPy [0.1, 0.3]                     일치
a1          손계산 [0.525, 0.5744]                NumPy [0.525, 0.5744]                일치
z2          손계산 0.6072                         NumPy 0.6072                         일치
y_pred      손계산 0.6473                         NumPy 0.6473                         일치
loss        손계산 0.435                          NumPy 0.435                          일치
dL/dy_pred  손계산 -1.5449                        NumPy -1.5449                        일치
dL/dz2      손계산 -0.3527                        NumPy -0.3527                        일치
dL/dW

## 5. 중심차분으로 기울기 검증 (gradient check)

역전파 공식이 맞는지 독립적으로 확인하기 위해, `src/calculus.py`의 `numerical_gradient`(중심차분)로 각 파라미터를 하나씩 ±h 움직여 손실의 기울기를 구한다.

$$
\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta + h e_i) - L(\theta - h e_i)}{2h}
$$

두 결과의 상대 오차 $\frac{\lVert g_{\text{역전파}} - g_{\text{수치}} \rVert}{\lVert g_{\text{역전파}} \rVert + \lVert g_{\text{수치}} \rVert}$ 가 1e-7 정도 이하면 역전파가 올바르다고 본다.

In [4]:
rel = gradient_check(grads)
assert rel < 1e-7

W1[0,0]  역전파 -0.04397852   중심차분 -0.04397852   차이 4.6e-12
W1[0,1]  역전파 -0.00000000   중심차분 +0.00000000   차이 0.0e+00
W1[1,0]  역전파 -0.05173350   중심차분 -0.05173350   차이 1.1e-11
W1[1,1]  역전파 -0.00000000   중심차분 +0.00000000   차이 0.0e+00
b1[0]    역전파 -0.04397852   중심차분 -0.04397852   차이 4.6e-12
b1[1]    역전파 -0.05173350   중심차분 -0.05173350   차이 1.1e-11
W2[0]    역전파 -0.18516459   중심차분 -0.18516459   차이 2.5e-13
W2[1]    역전파 -0.20261072   중심차분 -0.20261072   차이 3.3e-12
b2       역전파 -0.35270843   중심차분 -0.35270843   차이 7.2e-12

상대 오차 = 1.98e-11 → 역전파 공식 검증 완료


## 6. 정리

- 역전파는 출력에서 입력 방향으로 **연쇄 법칙**을 적용해, 각 단계의 국소 미분을 곱해 나가는 과정이다.
- sigmoid + BCE 조합에서는 $\partial L/\partial z_2 = \hat{y} - y$ 로 단순해진다. 예측이 정답보다 작으면(−0.3527) 기울기가 음수이고, 경사하강법은 기울기의 **반대** 방향으로 움직이므로 $W_2$ 를 **키우는** 쪽으로 업데이트한다.
- 각 기울기의 shape은 대응하는 변수의 shape과 같다. $\partial L/\partial W_1$ 은 $\partial L/\partial z_1$ (2,)과 $x$ (2,)의 외적으로 (2,2)가 된다.
- 입력이 0인 성분($x_2 = 0$)에 연결된 가중치의 기울기는 0이다.
- 손계산, NumPy, 중심차분 세 가지 방법의 결과가 모두 일치한다. 과제 PDF 예시의 $\partial L/\partial z_1$ 두 번째 값 −0.0513은 −0.0517의 오기로 보인다.